# ASTA - Remote Sensing Image Dehazing

This notebook tests pretrained weights and trains ASTA on the SateHaze1k-thick dataset.

**Dataset**: SateHaze1k-thick  
**Model**: ASTA (DehazeFormer variant)  
**Metrics**: PSNR, SSIM, SAM, ERGAS (validation) + comprehensive evaluation

In [ ]:
#@title 1. Install Dependencies
!pip install timm pytorch-msssim lpips scikit-image thop tensorboardX opencv-python

In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/ASTA_Results"
!mkdir -p "/content/drive/MyDrive/ASTA_Results/pretrained_results"
!mkdir -p "/content/drive/MyDrive/ASTA_Results/trained_results"
!mkdir -p "/content/drive/MyDrive/ASTA_Results/weights"

In [ ]:
#@title 3. Download SateHaze1k Dataset
!wget -O Haze1k.zip "https://www.dropbox.com/s/k2i3p7puuwl2g59/Haze1k.zip?dl=1"
!unzip -q Haze1k.zip -d ./Haze1k_dataset

!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/target/265.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/input/265.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/target/271.png
!rm -f ./Haze1k_dataset/Haze1k/Haze1k_moderate/dataset/train/input/271.png

!ls -l ./Haze1k_dataset/Haze1k/Haze1k_thick/dataset/

In [ ]:
#@title 4. Clone ASTA Repository
!git clone https://github.com/sumire25/ASTA.git
%cd ASTA
!ls

In [ ]:
#@title 5. Prepare Dataset Structure for ASTA
import os
import shutil

src_base = '/content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset'
dst_base = './data/Haze1k-thick'

os.makedirs(f'{dst_base}/train/input', exist_ok=True)
os.makedirs(f'{dst_base}/train/target', exist_ok=True)
os.makedirs(f'{dst_base}/test/input', exist_ok=True)
os.makedirs(f'{dst_base}/test/target', exist_ok=True)

for split in ['train', 'valid', 'test']:
    dst_split = 'test' if split == 'valid' else split
    for sub in ['input', 'target']:
        src_dir = os.path.join(src_base, split, sub)
        dst_dir = os.path.join(dst_base, dst_split, sub)
        if os.path.exists(src_dir):
            for f in os.listdir(src_dir):
                src_path = os.path.join(src_dir, f)
                dst_path = os.path.join(dst_dir, f)
                if not os.path.exists(dst_path):
                    shutil.copy2(src_path, dst_path)

print('Dataset prepared!')
!ls -l ./data/Haze1k-thick/train/input/ | head -5
!ls -l ./data/Haze1k-thick/test/input/ | head -5

In [ ]:
#@title 6. Compute FLOPs and Parameters
import torch
from thop import profile
from models import asta

network = asta().cuda()
network.eval()

dummy_input = torch.randn(1, 3, 256, 256).cuda()
macs, params = profile(network, inputs=(dummy_input,))

flops_g = macs / 1e9
params_m = params / 1e6

print(f"\n--- ASTA Complexity ---")
print(f"FLOPs: {flops_g:.4f} G")
print(f"Parameters: {params_m:.4f} M")

## Phase 1: Test Pretrained Weights

In [ ]:
#@title 7. Run Inference with Pretrained Weights
!python infer.py \
  --model asta \
  --weights "model weights/saved_models_SateHaze1k-TK/asta.pth" \
  --data_dir ./data/ \
  --dataset Haze1k-thick \
  --result_dir /content/drive/MyDrive/ASTA_Results/pretrained_results/

In [ ]:
#@title 8. Evaluate Pretrained Model
!python evaluate.py \
  --train_folder /content/drive/MyDrive/ASTA_Results/pretrained_results/ \
  --target_folder /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/

## Phase 2: Train from Scratch

In [ ]:
#@title 9. Train ASTA
!python train.py \
  --model asta \
  --data_dir ./data/ \
  --dataset Haze1k-thick \
  --save_dir /content/drive/MyDrive/ASTA_Results/weights/ \
  --log_dir ./logs/ \
  --num_workers 4 \
  --val_freq 3 \
  --gpu 0

## Phase 3: Test Trained Weights

In [ ]:
#@title 10. Run Inference with Trained Weights
!python infer.py \
  --model asta \
  --weights /content/drive/MyDrive/ASTA_Results/weights/asta.pth \
  --data_dir ./data/ \
  --dataset Haze1k-thick \
  --result_dir /content/drive/MyDrive/ASTA_Results/trained_results/

In [ ]:
#@title 11. Evaluate Trained Model
!python evaluate.py \
  --train_folder /content/drive/MyDrive/ASTA_Results/trained_results/ \
  --target_folder /content/Haze1k_dataset/Haze1k/Haze1k_thick/dataset/test/target/